In [1]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer

/Users/mahas/PycharmProjects/STIRS/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setting up the parameters
maximum_features = 30522  # Vocabulary size for BERT
maximum_length = 128  # Max length of input sequences
word_embedding_dims = 50  # Dimension of word embeddings (Note: In PyTorch this is embedding_dim)
no_of_filters = 128  # Number of filters for Conv1D
kernel_size = 3  # Size of the convolutional kernel
hidden_dim_1 = 128  # Neurons in the dense hidden layer

batch_size = 64
epochs = 10
threshold = 0.5
DATASET_SIZE = 1_000

# Load and preprocess the dataset
# Make sure the path to your CSV is correct
df = pd.read_csv("../jigsaw/dataset_text_target.csv")
df_true = df[df.target > 0.5]
df_false = df[df.target <= 0.5]
df = pd.concat([df_true[:DATASET_SIZE // 2], df_false[:DATASET_SIZE // 2]], axis=0)
mapper = lambda x: 1 if x > 0.5 else 0
df['target'] = df['target'].apply(mapper)

# Split data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(
    df.comment_text, df.target, test_size=0.2, random_state=42, shuffle=True
)

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if (torch.mps and torch.mps.is_available()) else "cpu")

In [4]:
# Tokenize and encode the data using the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

# Encode the training and test data
X_train_encoded = tokenizer.batch_encode_plus(
    x_train.tolist(),
    padding='max_length',
    truncation=True,
    max_length=maximum_length,
    add_special_tokens=True,
    return_tensors='pt',  # Return PyTorch tensors
)

X_test_encoded = tokenizer.batch_encode_plus(
    x_test.tolist(),
    padding='max_length',
    truncation=True,
    max_length=maximum_length,
    add_special_tokens=True,
    return_tensors='pt',  # Return PyTorch tensors
)

# Create PyTorch Datasets
train_dataset = TensorDataset(X_train_encoded['input_ids'], torch.tensor(y_train.values, dtype=torch.float32))
test_dataset = TensorDataset(X_test_encoded['input_ids'], torch.tensor(y_test.values, dtype=torch.float32))

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [5]:
from pytorch_transformers import BertForSequenceClassification

In [6]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1)
model.to(DEVICE)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [7]:
# train only the classification head, leave the BERT parameters alone

for param in model.bert.parameters():
    param.requires_grad = False

for param in model.bert.encoder.layer[-1].parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

In [8]:
def weighted_average(nums, weights):
    return sum(x[0]*x[1] for x in zip(nums, weights)) / sum(weights)

In [13]:
# Loss function and optimizer
criterion = nn.BCEWithLogitsLoss()

# Adam, consider only the classification head for the gradient
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# To store history
history = {
    'loss': [],
    'val_loss': [],
    'accuracy': [],
    'val_accuracy': [],
}

best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())

print("Starting training")
for epoch in range(epochs):
    # --- Training Phase ---
    model.train()
    total_loss = 0

    accuracies = []
    sizes = []

    for i, (input_ids, labels) in enumerate(train_loader):
        input_ids, labels = input_ids.to(DEVICE), labels.to(DEVICE).unsqueeze(1)

        print("\rBatch", i + 1, "of", len(train_loader), end="")
        if i+1 == len(train_loader): print()

        optimizer.zero_grad()
        outputs = model(input_ids)[0]

        accuracies.append(accuracy_score(torch.sigmoid(outputs).cpu() > threshold, labels.cpu()))
        sizes.append(len(labels))

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    avg_train_acc = weighted_average(accuracies, sizes)

    # --- Validation Phase ---
    model.eval()
    total_val_loss = 0

    accuracies = []
    sizes = []

    with torch.no_grad():
        for input_ids, labels in test_loader:
            input_ids, labels = input_ids.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
            outputs = model(input_ids)[0]

            accuracies.append(accuracy_score(torch.sigmoid(outputs).cpu() > threshold, labels.cpu()))
            sizes.append(len(labels))

            loss = criterion(outputs, labels)
            total_val_loss += loss.item()


    avg_val_acc = weighted_average(accuracies, sizes)
    avg_val_loss = total_val_loss / len(test_loader)

    # Save history
    history['loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['accuracy'].append(avg_train_acc)
    history['val_accuracy'].append(avg_val_acc)


    # Save the best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"\rEpoch {epoch + 1}/{epochs} | "
          f"Train Acc: {avg_train_acc:.4f} | "
          f"Val Acc: {avg_val_acc:.4f} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}")

# Load best model weights
model.load_state_dict(best_model_wts)

Starting training
Batch 13 of 13
Epoch 1/10 | Train Acc: 0.6362 | Val Acc: 0.6750 | Train Loss: 0.6328 | Val Loss: 0.5879
Batch 13 of 13
Epoch 2/10 | Train Acc: 0.7887 | Val Acc: 0.7500 | Train Loss: 0.5068 | Val Loss: 0.5072
Batch 13 of 13
Epoch 3/10 | Train Acc: 0.8413 | Val Acc: 0.7600 | Train Loss: 0.3819 | Val Loss: 0.5098
Batch 13 of 13
Epoch 4/10 | Train Acc: 0.8438 | Val Acc: 0.7500 | Train Loss: 0.3440 | Val Loss: 0.5053
Batch 13 of 13
Epoch 5/10 | Train Acc: 0.8850 | Val Acc: 0.7850 | Train Loss: 0.2806 | Val Loss: 0.4241
Batch 13 of 13
Epoch 6/10 | Train Acc: 0.9150 | Val Acc: 0.7800 | Train Loss: 0.2204 | Val Loss: 0.4632
Batch 13 of 13
Epoch 7/10 | Train Acc: 0.9275 | Val Acc: 0.8050 | Train Loss: 0.2021 | Val Loss: 0.3771
Batch 13 of 13
Epoch 8/10 | Train Acc: 0.9475 | Val Acc: 0.8150 | Train Loss: 0.1603 | Val Loss: 0.4201
Batch 13 of 13
Epoch 9/10 | Train Acc: 0.9637 | Val Acc: 0.8500 | Train Loss: 0.1198 | Val Loss: 0.3427
Batch 13 of 13
Epoch 10/10 | Train Acc: 0.9712

<All keys matched successfully>

In [9]:
model.eval()
y_pred_prob = []
y_true = []

with torch.no_grad():
    for input_ids, labels in test_loader:
        input_ids = input_ids.to(DEVICE)

        outputs = torch.sigmoid(model(input_ids)[0]).squeeze()

        y_pred_prob.extend(outputs.cpu().numpy())
        y_true.extend(labels.cpu().numpy())

y_pred = (np.array(y_pred_prob) > threshold).astype(int)

# Calculating and printing evaluation metrics
print('\nEvaluation Metrics:')
print('Accuracy:', accuracy_score(y_true, y_pred))
print('Precision:', precision_score(y_true, y_pred))
print('Recall:', recall_score(y_true, y_pred))
print('F1-score:', f1_score(y_true, y_pred))
print('\nClassification Report:')
print(classification_report(y_true, y_pred))


Evaluation Metrics:
Accuracy: 0.5
Precision: 0.4772727272727273
Recall: 0.4375
F1-score: 0.45652173913043476

Classification Report:
              precision    recall  f1-score   support

         0.0       0.52      0.56      0.54       104
         1.0       0.48      0.44      0.46        96

    accuracy                           0.50       200
   macro avg       0.50      0.50      0.50       200
weighted avg       0.50      0.50      0.50       200

